In [ ]:
#trying ml(not used in project)
import pandas as pd

df=pd.read_csv("../OpenActive_Merged.csv")
print(df.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

train_df=df.dropna(subset=["activity_raw","activity_type"]).copy()
train_df=train_df[train_df["activity_type"] != "Other / Unspecified"]

print(train_df.shape)
print(train_df["activity_type"].value_counts())

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(train_df["activity_raw"],train_df["activity_type"],test_size=0.2,random_state=42,stratify=train_df["activity_type"])
print(len(X_train),len(X_test))

In [ ]:
vectorizer=TfidfVectorizer(ngram_range=(1,2),max_features=2000)
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)

clf=LogisticRegression(max_iter=1000,class_weight="balanced")
clf.fit(X_train_vec,y_train)

In [ ]:
preds=clf.predict(X_test_vec)
print(classification_report(y_test,preds))

In [ ]:
other_rows=df[df["activity_type"] == "Other / Unspecified"].dropna(subset=["activity_raw"])
other_vec=vectorizer.transform(other_rows["activity_raw"])
predictions=clf.predict(other_vec)
probabilities=clf.predict_proba(other_vec).max(axis=1)

other_rows=other_rows.copy()
other_rows["predicted_type"]=predictions
other_rows["confidence"]=probabilities

reliable=other_rows[(other_rows["confidence"] > 0.7) &(~other_rows["predicted_type"].isin(["Dance","Wellness & Facility Access"]))]
print(reliable[["activity_raw","predicted_type","confidence"]].sample(20,random_state=1))

In [ ]:
#clustering
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

gap=pd.read_csv("../GAP_data/gap_scores_equity.csv")
profile=pd.read_csv("../GAP_data/borough_profile.csv")

data=gap.merge(profile[["borough","top_venue_share","avg_price","pct_better","blended_avg_price","pct_free"]],on="borough",how="left")
features=["demand_score_shrunk","sessions","venues","equity_gap","top_venue_share","blended_avg_price","pct_better"]
X=data[features].fillna(0)
print(X.describe())

In [ ]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

# checking elbow
inertias=[]
for k in range(2,8):
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

for k,i in zip(range(2,8),inertias):
    print(k,round(i,1))

In [ ]:
# silhouette score and Davies-Bouldin 
# to check whether if k=3 if good
from sklearn.metrics import silhouette_score,davies_bouldin_score

rows=[]
cluster_labels_by_k={}
for k in range(2,8):
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    labels=km.fit_predict(X_scaled)
    rows.append({"k": k,"inertia": km.inertia_,"silhouette": silhouette_score(X_scaled,labels),
        "davies_bouldin": davies_bouldin_score(X_scaled,labels),})
    cluster_labels_by_k[k]=labels

metrics_table=pd.DataFrame(rows)
print(metrics_table.round(4).to_string(index=False))

best_sil_k=int(metrics_table.loc[metrics_table["silhouette"].idxmax(),"k"])
best_db_k=int(metrics_table.loc[metrics_table["davies_bouldin"].idxmin(),"k"])
print(f"\nbest k by silhouette (higher=better): {best_sil_k}")
print(f"best k by DB (lower=better): {best_db_k}")

for k in (best_sil_k,best_db_k):
    labels=cluster_labels_by_k[k]
    print(f"\ncluster sizes at k={k}:")
    print(pd.Series(labels).value_counts().sort_index())

In [ ]:
import os
os.environ["OMP_NUM_THREADS"]="1"

km=KMeans(n_clusters=4,random_state=42,n_init=10)
data["cluster"]=km.fit_predict(X_scaled)

print(data.groupby("cluster")[features].mean())
print(data.groupby("cluster")["borough"].apply(list))

In [ ]:
pd.set_option("display.max_colwidth",None)
print(data.groupby("cluster")["borough"].apply(list))

In [ ]:
import os
os.environ["OMP_NUM_THREADS"]="1"
pd.set_option("display.max_colwidth",None)

for k in [3,4,5]:
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    data[f"cluster_k{k}"]=km.fit_predict(X_scaled)
    print(f"\n=== k = {k} ===")
    print(data.groupby(f"cluster_k{k}")[features].mean().round(2))
    print(data.groupby(f"cluster_k{k}")["borough"].apply(list))

In [ ]:
#k=5 has better silhouette score so kept
data["cluster"]=data["cluster_k5"]
cluster_names={0: "data-quality limited (Better-dependent)",1: "single-venue-dependent",2: "broad reliable supply",3: "genuine high need",4: "Haringey data quirk (price outlier)",}
data["cluster_label"]=data["cluster"].map(cluster_names)
data.to_csv("../GAP_data/gap_scores_clustered.csv",index=False)
print(data[["borough","class","cluster_label"]].sort_values("cluster_label"))

In [ ]:
# doing a check to see if this changes, changing into different fits
from scipy.optimize import linear_sum_assignment

def align_labels(labels,ref_labels,k):
    cost=np.zeros((k,k))
    for i in range(k):
        for j in range(k):
            cost[i,j]=-np.sum((labels == i) & (ref_labels == j))
    row_ind,col_ind=linear_sum_assignment(cost)
    mapping={row: col for row,col in zip(row_ind,col_ind)}
    return np.array([mapping[l] for l in labels])
import numpy as np
ref_labels=data["cluster"].values
boroughs=data["borough"].values
change_counts={b: 0 for b in boroughs}

for seed in range(10):
    km_seed=KMeans(n_clusters=5,random_state=seed,n_init=10)
    labels_seed=km_seed.fit_predict(X_scaled)
    aligned=align_labels(labels_seed,ref_labels,5)
    for b in boroughs[aligned != ref_labels]:
        change_counts[b] += 1

instability=pd.Series(change_counts).sort_values(ascending=False)
print("boroughs whose cluster changes across 10 different random_state values ")
print(instability[instability > 0])
print(instability[["Hounslow","Waltham Forest","Barking and Dagenham"]])

In [ ]:
# check clustering works is pct_better removed
features_no_better=["demand_score_shrunk","sessions","venues","equity_gap","top_venue_share","blended_avg_price"]
X_no_better=data[features_no_better].fillna(0)
X_no_better_scaled=StandardScaler().fit_transform(X_no_better)

km_no_better=KMeans(n_clusters=5,random_state=42,n_init=10)
labels_no_better=km_no_better.fit_predict(X_no_better_scaled)
aligned_no_better=align_labels(labels_no_better,ref_labels,5)

data["label_no_better_aligned"]=pd.Series(aligned_no_better).map(cluster_names).values
moved=data[data["cluster"] != aligned_no_better]
print(f"boroughs whose cluster changes when pct_better is dropped: {len(moved)} / {len(data)}")
print(moved[["borough","cluster_label","label_no_better_aligned"]].sort_values("cluster_label").to_string(index=False))
print("cluster sizes with pct_better:")
print(data["cluster_label"].value_counts())
print("cluster sizes without pct_better ")
print(data["label_no_better_aligned"].value_counts())

In [ ]:
#why clusetring differs
mismatch=data[((data["class"].isin(["genuine desert","blind spot","under-monitored","emerging desert"])) & (data["cluster_label"] != "genuine high need")) |((data["class"].isin(["well-served","well-served (moderate need)"])) & (data["cluster_label"] == "genuine high need"))]
print(mismatch[["borough","class","cluster_label"]])

In [17]:
print(data.groupby("cluster_label")["equity_gap"].agg(["mean","std","count"]))

                                              mean       std  count
cluster_label                                                      
Haringey data quirk (price outlier)       4.664043       NaN      1
broad reliable supply                     5.886758  3.600568      7
data-quality limited (Better-dependent)   4.785951  2.776103     10
genuine high need                         4.274758  2.945475     10
single-venue-dependent                   12.123118  5.129525      4


In [18]:
print(data[data["cluster_label"]=="broad reliable supply"][["borough","equity_gap"]])

           borough  equity_gap
2           Bexley   10.186206
7           Ealing    1.750107
14        Havering    4.958853
26       Southwark    3.891041
29  Waltham Forest    4.073767
30      Wandsworth   11.608535
31     Westminster    4.738799
